In [2]:
import os, tempfile, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterSampler, cross_val_score
from sklearn.base import clone
from tqdm.auto import tqdm

from xgboost import XGBClassifier

import mlflow

c:\Users\aman0\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Loading and Preprocessing

In [3]:
tournaments_df = pd.read_csv('../Data/Staging/tournaments.csv').set_index(['Name', 'Year'])
players_df = pd.read_csv('../Data/Staging/players.csv').set_index('Name')

df = pd.read_csv('../Data/Staging/feature_table.csv')
df = df.drop(df.columns[[0]], axis = 1)
df = df.drop(['Player 1', 'Player 2', 'Tournament Name', 'Year', 'Start Date', 'End Date'], axis = 1)
X = df.drop('Winner', axis = 1)
y = df['Winner']

MLFlow run to select best model based on cv accuracy (since classes are balanced we need not worry about other metrics like precision, recall, or f1-score)

In [ ]:
# -----------------------------
# Utilities for plotting artifacts
# -----------------------------
def save_confusion_matrix(y_true, y_pred, out_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(title="Confusion Matrix", xlabel="Predicted", ylabel="True")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["0","1"]); ax.set_yticklabels(["0","1"])
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_roc_curve(y_true, y_score, out_path):
    fig, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y_true, y_score, ax=ax)
    ax.set_title("ROC Curve")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_pr_curve(y_true, y_score, out_path):
    fig, ax = plt.subplots()
    PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=ax)
    ax.set_title("Precision-Recall Curve")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_feature_importance(importances, feature_names, out_path_png, out_path_csv, top_k=30):
    df = pd.DataFrame({"feature": feature_names, "importance": importances})
    df = df.sort_values("importance", ascending=False)
    df.to_csv(out_path_csv, index=False)
    take = df.head(top_k)
    fig, ax = plt.subplots()
    ax.barh(take["feature"][::-1], take["importance"][::-1])
    ax.set_title("Feature Importance")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    fig.savefig(out_path_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# Model candidates
# -----------------------------
def _ohe(**kwargs):
    # Back-compat for sklearn <1.2 (sparse) vs >=1.2 (sparse_output)
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, **kwargs)

def get_candidates(X, categorical_cols=('Round','Surface')):
    candidates = []
    cat_cols = list(categorical_cols)
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', _ohe(), cat_cols),
            ('num', StandardScaler(), num_cols)
        ]
    )

    # Logistic Regression
    pipe_lr = Pipeline([
        ('preprocess', preprocessor),
        ('clf', LogisticRegression(max_iter=500, solver='lbfgs'))
    ])
    param_lr = {
        'clf__C': np.logspace(-3, 2, 20),
        'clf__penalty': ['l2'],
    }
    candidates.append(("logreg", pipe_lr, param_lr))

    # Random Forest
    pipe_rf = Pipeline([
        ('preprocess', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=400, n_jobs=-1, class_weight=None, random_state=42
        ))
    ])
    param_rf = {
        'clf__max_depth': [None, 4, 6, 8, 12, 16],
        'clf__min_samples_split': [2, 5, 10, 20],
        'clf__min_samples_leaf': [1, 2, 4, 8],
        'clf__max_features': ['sqrt', 'log2', 0.5, None],
    }
    candidates.append(("random_forest", pipe_rf, param_rf))

    # XGBoost
    pipe_xgb = Pipeline([
        ('preprocess', preprocessor),
        ('clf', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            random_state=42,
            n_estimators=500,
            n_jobs=-1,
            verbosity=0
        ))
    ])
    param_xgb = {
        'clf__max_depth': [3,4,5,6,7,8],
        'clf__learning_rate': np.linspace(0.01, 0.3, 10).tolist(),
        'clf__subsample': np.linspace(0.6, 1.0, 5).tolist(),
        'clf__colsample_bytree': np.linspace(0.6, 1.0, 5).tolist(),
        'clf__min_child_weight': [1,2,5,10],
        'clf__gamma': [0.0, 0.5, 1.0]
    }
    candidates.append(("xgboost", pipe_xgb, param_xgb))

    return candidates

def _short_params(d, max_len=110, keep_keys=4):
    if not d:
        return ""
    items = list(d.items())
    order = sorted(items, key=lambda kv: (0 if any(k in kv[0].lower() for k in [
        "C","max_depth","learning_rate","n_estimators","min_child_weight","gamma","subsample","colsample"]) else 1, kv[0]))
    s = ", ".join(f"{k}={v}" for k, v in order[:keep_keys])
    if len(order) > keep_keys: s += ", …"
    return (s if len(s) <= max_len else s[:max_len-1] + "…")

# -----------------------------
# Main search
# -----------------------------
def main(X, y):
    mlflow.set_experiment("model-search-test")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    candidates = get_candidates(X)
    n_iter = 30
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    best = {"name": None, "val_acc": -np.inf, "model": None, "params": None}

    for fam_idx, (name, estimator, param_grid) in enumerate(candidates, start=1):
        with mlflow.start_run(run_name=f"{name}-search"):
            sampler = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))
            family_best_score = -np.inf
            family_best_params = None
            family_best_est = None

            print(f"\n=== [{fam_idx}/{len(candidates)}] {name}: {n_iter} trials, {cv.get_n_splits()}-fold CV (accuracy) ===")
            print(f"{'trial':>5} | {'cv_acc':>10} | {'best_so_far':>10} | params")
            print("-" * 90)

            for trial_idx, params in enumerate(sampler, start=1):
                est = clone(estimator).set_params(**params)

                scores = cross_val_score(
                    est, X_train, y_train,
                    scoring="accuracy", cv=cv, n_jobs=-1
                )
                mean_acc = float(np.mean(scores))
                std_acc = float(np.std(scores))

                mlflow.log_metric("cv_acc_trial", mean_acc, step=trial_idx-1)

                param_str = _short_params(params)
                print(f"{trial_idx:5d} | {mean_acc:>.6f}±{std_acc:.4f} | {max(family_best_score, mean_acc):>.6f} | {param_str}", flush=True)

                if mean_acc > family_best_score:
                    family_best_score = mean_acc
                    family_best_params = params
                    family_best_est = clone(estimator).set_params(**params)

            family_best_est.fit(X_train, y_train)
            if hasattr(family_best_est, "predict_proba"):
                y_proba = family_best_est.predict_proba(X_test)[:, 1]
            else:
                y_proba = family_best_est.decision_function(X_test)
                y_proba = (y_proba - y_proba.min()) / (y_proba.ptp() + 1e-12)
            y_pred = (y_proba >= 0.5).astype(int)

            test_auc = roc_auc_score(y_test, y_proba)
            pr_auc = average_precision_score(y_test, y_proba)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred)
            rec = recall_score(y_test, y_pred)

            mlflow.log_param("model_family", name)
            mlflow.log_params({f"best__{k}": v for k, v in (family_best_params or {}).items()})
            mlflow.log_metric("cv_val_acc_mean", float(family_best_score))
            mlflow.log_metric("test_auc", float(test_auc))
            mlflow.log_metric("test_pr_auc", float(pr_auc))
            mlflow.log_metric("test_accuracy", float(acc))
            mlflow.log_metric("test_f1", float(f1))
            mlflow.log_metric("test_precision", float(prec))
            mlflow.log_metric("test_recall", float(rec))

            print(f"-> {name} best CV acc: {family_best_score:.6f} | test acc: {acc:.6f}")

            if family_best_score > best["val_acc"]:
                best.update({
                    "name": name,
                    "val_acc": float(family_best_score),
                    "model": family_best_est,
                    "params": family_best_params
                })

    # ---------- Final run for best model ----------
    winner = best["model"]
    name = best["name"]

    if hasattr(winner, "predict_proba"):
        y_proba = winner.predict_proba(X_test)[:, 1]
    else:
        y_proba = winner.decision_function(X_test)
        y_proba = (y_proba - y_proba.min()) / (y_proba.ptp() + 1e-12)
    y_pred = (y_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    with mlflow.start_run(run_name=f"best-model-{name}"):
        mlflow.log_param("selected_model_family", name)
        mlflow.log_params({f"selected__{k}": v for k, v in (best["params"] or {}).items()})
        mlflow.log_metric("cv_val_acc_mean", float(best["val_acc"]))
        mlflow.log_metric("test_auc", float(test_auc))
        mlflow.log_metric("test_pr_auc", float(pr_auc))
        mlflow.log_metric("test_accuracy", float(acc))
        mlflow.log_metric("test_f1", float(f1))
        mlflow.log_metric("test_precision", float(prec))
        mlflow.log_metric("test_recall", float(rec))

        with tempfile.TemporaryDirectory() as td:
            cm_path = os.path.join(td, "confusion_matrix.png")
            roc_path = os.path.join(td, "roc_curve.png")
            pr_path = os.path.join(td, "pr_curve.png")
            save_confusion_matrix(y_test, y_pred, cm_path)
            save_roc_curve(y_test, y_proba, roc_path)
            save_pr_curve(y_test, y_proba, pr_path)
            mlflow.log_artifact(cm_path, artifact_path="plots")
            mlflow.log_artifact(roc_path, artifact_path="plots")
            mlflow.log_artifact(pr_path,  artifact_path="plots")

            feat_names = list(X.columns) if hasattr(X, "columns") else [f"f{i}" for i in range(X.shape[1])]
            importances = None
            if hasattr(winner, "feature_importances_"):
                importances = np.asarray(winner.feature_importances_)
            elif hasattr(winner, "coef_") and getattr(winner, "coef_", None) is not None:
                importances = np.abs(np.ravel(winner.coef_))
            if importances is None:
                r = permutation_importance(winner, X_test, y_test, scoring="accuracy",
                                           n_repeats=10, n_jobs=-1, random_state=42)
                importances = r.importances_mean
            fi_png = os.path.join(td, "feature_importance.png")
            fi_csv = os.path.join(td, "feature_importance.csv")
            save_feature_importance(importances, feat_names, fi_png, fi_csv, top_k=30)
            mlflow.log_artifact(fi_png, artifact_path="feature_importance")
            mlflow.log_artifact(fi_csv, artifact_path="feature_importance")

            schema_path = os.path.join(td, "train_columns.json")
            with open(schema_path, "w") as f:
                json.dump({"columns": feat_names}, f, indent=2)
            mlflow.log_artifact(schema_path, artifact_path="schema")

        input_example = X_test[:2] if hasattr(X_test, "head") else X_test[:2]
        mlflow.sklearn.log_model(
            winner,
            artifact_path="model",
            input_example=input_example,
            registered_model_name=None
        )

    print("\n=== BEST MODEL ===")
    print({"name": best["name"], "cv_val_acc_mean": best["val_acc"]})
    print("Open MLflow UI: mlflow ui  (http://127.0.0.1:5000)")
if __name__ == "__main__":
    main(X,y)

c:\Users\aman0\miniconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 20 is smaller than n_iter=30. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



=== [1/3] logreg: 30 trials, 5-fold CV (accuracy) ===
trial |     cv_acc | best_so_far | params
------------------------------------------------------------------------------------------
    1 | 0.633188±0.0040 | 0.633188 | clf__C=0.001, clf__penalty=l2
    2 | 0.634056±0.0042 | 0.634056 | clf__C=0.0018329807108324356, clf__penalty=l2
    3 | 0.634883±0.0041 | 0.634883 | clf__C=0.003359818286283781, clf__penalty=l2
    4 | 0.635154±0.0037 | 0.635154 | clf__C=0.006158482110660267, clf__penalty=l2
    5 | 0.635805±0.0036 | 0.635805 | clf__C=0.011288378916846888, clf__penalty=l2
    6 | 0.636104±0.0038 | 0.636104 | clf__C=0.02069138081114789, clf__penalty=l2
    7 | 0.636497±0.0039 | 0.636497 | clf__C=0.0379269019073225, clf__penalty=l2
    8 | 0.636402±0.0038 | 0.636497 | clf__C=0.06951927961775606, clf__penalty=l2
    9 | 0.636375±0.0039 | 0.636497 | clf__C=0.12742749857031335, clf__penalty=l2
   10 | 0.636537±0.0040 | 0.636537 | clf__C=0.23357214690901212, clf__penalty=l2
   11 | 0.63